# PoC: grounding a summary back to transcript lines

Pipeline:

1. **Break the transcript into lines** (explicit line breaks, or sentences).
2. **Embed each line** with `sentence-transformers/all-mpnet-base-v2`.
3. **Summarize** the transcript with `google/pegasus-cnn_dailymail`, then
   **embed the summary with the *same* `all-mpnet-base-v2`**.
4. **Cosine-rank** the transcript lines by similarity to the summary.

> Why embed the summary with all-mpnet and not Pegasus: Pegasus is a *summarizer*,
> not a sentence-embedding model. For cosine similarity to mean anything, the
> lines and the summary must be embedded by the **same** model into the **same**
> vector space. So Pegasus produces the summary *text*; all-mpnet embeds both
> sides.

By default the "transcript" is one article from `test.csv`, but you can paste any
text into `TRANSCRIPT` below.

In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "sentence-transformers", "transformers", "torch", "pandas",
                "nltk", "sentencepiece", "protobuf"], check=True)

CompletedProcess(args=['/Users/angie/miniforge3/bin/python', '-m', 'pip', 'install', '-q', 'sentence-transformers', 'transformers', 'torch', 'pandas', 'nltk', 'sentencepiece', 'protobuf'], returncode=0)

In [2]:
import pandas as pd
import torch
import nltk
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util

for res, path in [("punkt", "tokenizers/punkt"), ("punkt_tab", "tokenizers/punkt_tab")]:
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(res, quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("Device:", DEVICE)

/Users/angie/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


## Config

In [4]:
PEGASUS_MODEL = "google/pegasus-cnn_dailymail"
EMBED_MODEL   = "sentence-transformers/all-mpnet-base-v2"

TEST_CSV   = "cnn_dailymail_summary/test.csv"
ROW_INDEX  = 0      # which article to use as the transcript
TRANSCRIPT = None   # paste your own transcript string here to override test.csv

TOP_K = 5           # how many best-matching lines to show

## 1. Break the transcript into lines

In [5]:
def split_lines(text):
    """Use explicit line breaks if present (real transcripts), else sentences."""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if len(lines) > 1:
        return lines
    return [s.strip() for s in nltk.sent_tokenize(text) if s.strip()]

if TRANSCRIPT is not None:
    transcript = TRANSCRIPT
else:
    transcript = pd.read_csv(TEST_CSV)["article"].iloc[ROW_INDEX]

lines = split_lines(transcript)
print(f"{len(lines)} lines\n")
for i, line in enumerate(lines[:8]):
    print(f"[{i}] {line}")

16 lines

[0] Ever noticed how plane seats appear to be getting smaller and smaller?
[1] With increasing numbers of people taking to the skies, some experts are questioning if having such packed out planes is putting passengers at risk.
[2] They say that the shrinking space on aeroplanes is not only uncomfortable - it's putting our health and safety in danger.
[3] More than squabbling over the arm rest, shrinking space on planes putting our health and safety in danger?
[4] This week, a U.S consumer advisory group set up by the Department of Transportation said at a public hearing that while the government is happy to set standards for animals flying on planes, it doesn't stipulate a minimum amount of space for humans.
[5] 'In a world where animals have more rights to space and food than humans,' said Charlie Leocha, consumer representative on the committee.
[6] 'It is time that the DOT and FAA take a stand for humane treatment of passengers.'
[7] But could crowding on planes lead to mo

## 2. Summarize the transcript with Pegasus

In [6]:
tok = AutoTokenizer.from_pretrained(PEGASUS_MODEL)
tok.model_max_length = 1024
pegasus = AutoModelForSeq2SeqLM.from_pretrained(PEGASUS_MODEL).to(DEVICE).eval()

inputs = tok(transcript, max_length=1024, truncation=True, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    ids = pegasus.generate(**inputs, max_length=128, min_length=30, num_beams=4, do_sample=False)
summary = tok.batch_decode(ids, skip_special_tokens=True)[0].replace("<n>", " ").strip()
print("SUMMARY:\n", summary)

Loading weights: 100%|██████████| 680/680 [00:00<00:00, 58818.86it/s]
[transformers] PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-cnn_dailymail
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SUMMARY:
 U.S consumer advisory group set up by Department of Transportation . Says government doesn't stipulate minimum amount of space for humans . Tests conducted by FAA use planes with 31 inch pitch, a standard which on some airlines has decreased .


## 3. Embed lines + summary with all-mpnet-base-v2

`normalize_embeddings=True` makes the dot product equal to cosine similarity.

In [7]:
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)

line_emb = embedder.encode(lines, convert_to_tensor=True, normalize_embeddings=True)
summary_emb = embedder.encode(summary, convert_to_tensor=True, normalize_embeddings=True)
print("line embeddings:", tuple(line_emb.shape), "| summary embedding:", tuple(summary_emb.shape))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9216.83it/s]


line embeddings: (16, 768) | summary embedding: (768,)


## 4. Rank transcript lines by similarity to the summary

In [8]:
sims = util.cos_sim(summary_emb, line_emb)[0]
order = sims.argsort(descending=True)

ranked = pd.DataFrame({
    "rank": range(1, len(order) + 1),
    "score": [round(sims[i].item(), 4) for i in order],
    "line": [lines[i] for i in order],
})
print(f"Top {TOP_K} transcript lines most similar to the summary:\n")
ranked.head(TOP_K)

Top 5 transcript lines most similar to the summary:



,rank,score,line
0,1,0.7317,"This week, a U.S consumer advisory group set u..."
1,2,0.7290,While most airlines stick to a pitch of 31 inc...
2,3,0.7151,Tests conducted by the FAA use planes with a 3...
3,4,0.6827,But these tests are conducted using planes wit...
4,5,0.6468,"While United Airlines has 30 inches of space, ..."
